In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('train.txt',sep = ';',header = None,names = ['text','emotion'])

In [3]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
df.isnull().sum()

,0
text,0
emotion,0


In [5]:
unique_emotions = df['emotion'].unique()
emotion_numbers = {}
i = 0
for emo in unique_emotions:
  emotion_numbers[emo] = i
  i +=1

df['emotion'] = df['emotion'].map(emotion_numbers)

In [6]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [7]:
df['text'] = df['text'].apply(lambda x : x.lower())

In [8]:
import string

def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))

In [9]:
df['text'] = df['text'].apply(remove_punc)

In [10]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

In [11]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

In [12]:
import nltk

In [13]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [14]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [15]:
stop_words = set(stopwords.words('english'))

In [16]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [17]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)

In [18]:
df['text'] = df['text'].apply(remove)

In [19]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [20]:
# from sklearn.feature_extraction.text import CountVectorizer

# documents = [
#     'i love pizza',
#     'Pizza is the best',
#     'I love pasta',
#     'Pasta is great'
# ]

# vectorizer = CountVectorizer(ngram_range=(3,3))
# X = vectorizer.fit_transform(documents)

# print('Vocabulary:', vectorizer.get_feature_names_out())
# print('\nBoW Matrix:\n', X.toarray())

In [21]:
# from sklearn.feature_extraction.text import TfidfVectorizer

# documents = [
#     'i love pizza',
#     'Pizza is the best',
#     'I love pasta',
#     'Pasta is great'
# ]

# vectorizer2 = TfidfVectorizer()
# X = vectorizer2.fit_transform(documents)

# print('Vocabulary:', vectorizer2.get_feature_names_out())
# print('\nBoW Matrix:\n', X.toarray())

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.33, random_state=42)

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

pred_bow = nb_model.predict(X_test_bow)
print("Accuracy:", accuracy_score(y_test, pred_bow))

Accuracy: 0.7647727272727273


In [23]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf, y_train)

pred_tfidf = nb_model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, pred_tfidf))

Accuracy: 0.7231060606060606


In [24]:
from sklearn.linear_model import LogisticRegression

In [25]:
logistic_model = LogisticRegression(max_iter=1000)

In [26]:
logistic_model.fit(X_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In [27]:
log_pred = logistic_model.predict(X_test_tfidf)

In [28]:
print(accuracy_score(log_pred,y_test))

0.8479166666666667


In [29]:
from sklearn.svm import SVC

In [30]:
svm_model = SVC(kernel='linear',gamma='scale',probability=True,class_weight='balanced')

In [31]:
svm_model.fit(X_train_tfidf,y_train)

SVC(class_weight='balanced', kernel='linear', probability=True)

In [32]:
svm_pred = svm_model.predict(X_test_tfidf)

In [33]:
print(accuracy_score(svm_pred,y_test))

0.8804924242424242


In [34]:
import joblib

In [36]:
joblib.dump((svm_model, tfidf_vectorizer), "svm_model.pkl")

['svm_model.pkl']